# Лабораторная работа №1

Задания:
  - найти велосипед с максимальным временем пробега
  - найти наибольшей геодезическое расстояние между станциями
  - найти путь велосипеда с максимальным временем пробега через станции
  - найти количество велосипедов в системе
  - найти пользователей потративших на поездки более 3 часов


In [1]:
!pip install pyspark

In [22]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as t
import math

In [7]:
spark = SparkSession.builder.getOrCreate()

***Считываем данные***

In [8]:
trips = spark.read.format('csv').option('header', 'true').option('inferSchema', 'true').load("/content/drive/MyDrive/Учёба/8 семестр/БД/trip.csv")
stations = spark.read.format('csv').option('header', 'true').option('inferSchema', 'true').load("/content/drive/MyDrive/Учёба/8 семестр/БД/station.csv")

***1. Велосипед с максимальным временем пробега***

In [14]:
bike_max_total_time = trips.groupBy('bike_id') \
    .agg(F.sum("duration").alias("total_duration")) \
    .orderBy(F.col("total_duration").desc())

top_bike = bike_max_total_time.first()
print(f"Велосипед {top_bike['bike_id']} проехал максимум {top_bike['total_duration']} сек")

Велосипед 535 проехал максимум 18611693 сек


***2. Наибольшее геодезическое расстояние (Формула Гаверсинуса)***



In [15]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371 # радиус Земли
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon/2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))
    return R * c

# регистрация функции как UDF для Spark
haversine_udf = F.udf(haversine, t.DoubleType())

stations_data = stations.select("id", "lat", "long")
# кросс-соединение (все со всеми)
dist_df = stations_data.alias("a").join(stations_data.alias("b"), F.col("a.id") < F.col("b.id")) \
    .withColumn("distance", haversine_udf(F.col("a.lat"), F.col("a.long"), F.col("b.lat"), F.col("b.long")))

max_dist = dist_df.orderBy(F.col("distance").desc()).first()
print(f"2. Наибольшее геодезическое расстояние: {max_dist['distance']:.2f} км")

2. Наибольшее геодезическое расстояние: 69.92 км


***3. Путь велосипеда с максимальным временем пробега через станции***

In [16]:
# ID из пункта 1
path = trips.filter(F.col("bike_id") == top_bike['bike_id']) \
    .orderBy("start_date") \
    .select("start_station_name", "end_station_name")

print(f"3. Путь велосипеда {top_bike['bike_id']}:")
path.show(truncate=False)

3. Путь велосипеда 535:
+---------------------------------------------+---------------------------------------------+
|start_station_name                           |end_station_name                             |
+---------------------------------------------+---------------------------------------------+
|Mechanics Plaza (Market at Battery)          |Embarcadero at Sansome                       |
|Embarcadero at Sansome                       |Market at 4th                                |
|Market at 4th                                |South Van Ness at Market                     |
|Market at 10th                               |Powell Street BART                           |
|Embarcadero at Folsom                        |San Francisco Caltrain (Townsend at 4th)     |
|San Francisco Caltrain (Townsend at 4th)     |Temporary Transbay Terminal (Howard at Beale)|
|Temporary Transbay Terminal (Howard at Beale)|Market at 10th                               |
|Powell Street BART                 

***4. Количество велосипедов в системе***

In [13]:
count_bikes = trips.select("bike_id").distinct().count()
print(f"4. Всего велосипедов: {count_bikes}")

4. Всего велосипедов: 700


***5. Пользователи, потратившие на поездки более 3 часов (то есть 10800 сек)***

In [25]:
# группировка по zip_code и сумма всех поездок
output5 = (
    trips
    .groupBy('zip_code')
    .agg(
        F.sum(F.col("duration").cast(t.IntegerType())).alias("total_duration")
    )
)

# фильтруем тех, кто наездил больше 3 часов
output5 = output5.filter(F.col("total_duration") >= 10800).orderBy(F.col("total_duration").desc())

print("5. Пользователи, потратившие на поездки более 3 часов:")
output5.show(20, truncate=False)

# вывести прям весь список таких пользователей:
# output5.show(n=output5.count(), truncate=False)

5. Пользователи, потратившие на поездки более 3 часов:
+--------+--------------+
|zip_code|total_duration|
+--------+--------------+
|94107   |49757162      |
|nil     |45725550      |
|NULL    |27723273      |
|94105   |25596128      |
|94133   |21637675      |
|94102   |19128021      |
|94103   |19127388      |
|95531   |17270400      |
|94111   |14244997      |
|95112   |12742370      |
|94109   |12057128      |
|94040   |7807926       |
|94110   |7421936       |
|94117   |6901313       |
|94301   |6590378       |
|94041   |6276284       |
|94158   |6248167       |
|94306   |5550643       |
|94025   |5178237       |
|94108   |5127562       |
+--------+--------------+
only showing top 20 rows
